# JW $Z_4^{TF}$ system

Created: 03-09-2026

Objectives:
* Iterate on [this notebook](jw_z_4_tf_system.ipynb), using a more advanced optimization to find the projectors. Take 3 adjacent, disjoint regions $A, P, B$. For a given $\ket{v}$ supported in $P$, define $\rho_{AB} = \Tr_{(A \cup B)^c}\braket{v | \rho | v}$, and $\rho_A, \rho_B$ defined accordingly. We wish to find a $\ket{v}$ such that $\rho_{AB} = \rho_A \otimes \rho_B$. Thus the optimization norm we use is $$||\rho_{AB} - \rho_A \otimes \rho_B||_{Fr}$$, i.e. the trace norm squared of $\rho_{AB} - \rho_A \otimes \rho_B$. This reduces to $$\Tr(\rho_{AB}^2) + \Tr(\rho_A^2)\Tr(\rho_B^2) - 2\Tr(\rho(\rho_A \otimes \rho_B))$$
* Also need to ensure $\ket{v}$ respects fermion parity symmetry.

# Imports

In [1]:
import numpy as np

In [2]:
import jax
jax.config.update('jax_platform_name', 'cpu')

import jax.numpy as jnp

In [3]:
import matplotlib.pyplot as plt

In [4]:
from tqdm import tqdm

In [5]:
from functools import reduce
from operator import mul

In [6]:
from random import random

In [7]:
import quimb.tensor as qtn
import quimb as qu

In [8]:
from scipy.stats import unitary_group

In [9]:
from collections import Counter

In [10]:
import pandas as pd

In [11]:
from time import time

In [12]:
from humanize import naturalsize

# Definitions
## Construct cluster state

In [13]:
np_up_X_state = 1/(np.sqrt(2))*np.array([1,1])

In [14]:
qu_up_X_state = qtn.Tensor(
    data=np_up_X_state,
    inds=('k',),
    tags='prod'
)

In [15]:
np_CZ = np.diag([1,1,1,-1])

In [16]:
np_CZ = np_CZ.reshape((2,)*4)

In [17]:
qu_CZ = qtn.Tensor(
    data=np_CZ,
    inds=('k1', 'k2', 'b1', 'b2'),
    tags='CZ'
)

In [18]:
np_hadamard = np.pow(2, -1/2)*np.array([
    [1,1],
    [1,-1]
])

In [19]:
qu_hadamard = qtn.Tensor(data=np_hadamard, inds=('k', 'b'), tags='Had')

In [20]:
def get_cluster_state_qu_tensor_network(num_sites):
    assert (num_sites%2) == 0

    product_state_tensors = [
        qu_up_X_state.reindex({'k': f'kc_1_{i}'})
        for i in range(num_sites)
    ]

    first_layer_circuit_tensors = [
        qu_CZ.reindex({
            'b1': f'kc_1_{i}',
            'b2': f'kc_1_{i+1}',
            'k1': f'kc_2_{i}',
            'k2': f'kc_2_{i+1}'
        })
        for i in range(0, num_sites, 2)
    ]


    second_layer_circuit_tensors = [
        qu_CZ.reindex({
            'b1': f'kc_2_{i}',
            'b2': f'kc_2_{(i+1)%num_sites}',
            'k1': f'kh_{i}',
            'k2': f'k{(i+1)%num_sites}'
        })
        for i in range(1, num_sites+1, 2)
    ]

    hadamard_layer = [
        qu_hadamard.reindex({
            'b': f'kh_{i}',
            'k': f'k{i}'
        })
        for i in range(1, num_sites, 2)
    ]
    all_tensors = (
        product_state_tensors
        + first_layer_circuit_tensors
        + second_layer_circuit_tensors
        + hadamard_layer
    )

    out = qtn.TensorNetwork(all_tensors, virtual=True)
    out.mangle_inner_()

    return out

## Construct product state

In [21]:
np_up_X_state = 1/(np.sqrt(2))*np.array([1,1])

In [22]:
np_up_Z_state = np.array([1,0])

In [23]:
qu_up_Z_state = qtn.Tensor(
    data=np_up_Z_state,
    inds=('k',),
    tags='prod'
)

In [24]:
alternating_states = [
    qu_up_X_state,
    qu_up_Z_state
]

def get_product_qu_tensor_network(num_sites):
    assert (num_sites%2) == 0

    product_state_tensors = [
        alternating_states[i%2].reindex({'k': f'k{i}'})
        for i in range(num_sites)
    ]

    out = qtn.TensorNetwork(
        product_state_tensors,
        virtual=True
    )
    out.mangle_inner_()

    return out

## Symmetries

In [25]:
def multikron(arrays):
    return reduce(np.kron, arrays)

In [26]:
np_I = np.array([
    [1,0],
    [0,1]
])

np_X = np.array([
    [0,1],
    [1,0]
])

np_Y = np.array([
    [0,-1j],
    [1j,0]
])

np_Z = np.array([
    [1,0],
    [0,-1]
])

In [27]:
qu_I = qtn.Tensor(
    np_I,
    inds=['k', 'b'],
    tags='X'
)

qu_X = qtn.Tensor(
    np_X,
    inds=['k', 'b'],
    tags='X'
)

qu_Y = qtn.Tensor(
    np_Y,
    inds=['k', 'b'],
    tags='Y'
)

qu_Z = qtn.Tensor(
    np_Z,
    inds=['k', 'b'],
    tags='Z'
)

In [28]:
def get_multisite_qu_X(num_sites):
    np_many_X = multikron([np_X]*num_sites)

    out = qtn.Tensor(
        np_many_X,
        inds=['k', 'b'],
        tags='mulit_site_X',
    )

    return out

In [29]:
def get_multisite_qu_I(num_sites):
    np_many_I = multikron([np_I]*num_sites)

    out = qtn.Tensor(
        np_many_I,
        inds=['k', 'b'],
        tags='mulit_site_I',
    )

    return out

In [30]:
qu_spin_fermion_fp = (
    qu_I.reindex({'k': 'ks', 'b': 'bs'})
    & qu_Z.reindex({'k': 'kf', 'b': 'bf'})
).contract()

qu_unit_cell_fp = qu_spin_fermion_fp.fuse({
    'k': ['ks', 'kf'],
    'b': ['bs', 'bf']
})

In [31]:
"""
def get_multisite_qu_fp(num_sites):
    assert (num_sites%2)==0

    num_unit_cells = (num_sites//2)

    np
"""

'\ndef get_multisite_qu_fp(num_sites):\n    assert (num_sites%2)==0\n\n    num_unit_cells = (num_sites//2)\n\n    np\n'

Decompose T symmetry as $MK$:

In [32]:
np_00 = np.array([[1,0], [0,0]])
np_11 = np.array([[0,0], [0,1]])

In [33]:
def tensor_product_operators(op_1, op_2):
    out = (
        op_1[..., np.newaxis, np.newaxis]
        *op_2[np.newaxis, np.newaxis, ...]
    )

    return out

In [34]:
np_M = (
    tensor_product_operators(np_X, np_00)
    + tensor_product_operators(np_Y, np_11)
)

In [35]:
qu_M = qtn.Tensor(
    np_M,
    inds=['ks', 'bs', 'kf', 'bf']
)

In [36]:
np_M_reindexed = (
    qu_M
    .fuse({
        'k': ['ks', 'kf'],
        'b': ['bs', 'bf']
    })
    .transpose('k', 'b')
    .data
)

## Extracting projectors

In [37]:
def random_uniform_complex(shape):
    return np.random.uniform(size=shape) + 1j*np.random.uniform(size=shape)

In [38]:
def maximize_projector_states(rho, left_sites, proj_sites, right_sites):
    v = qtn.Tensor(
        data=random_uniform_complex((2,)*(len(proj_sites)-1)),
        inds=[f'k{i}' for i in proj_sites[:-1]]
    )

    tnopt = qtn.TNOptimizer(
        v,  # the tensor network we want to optimize
        loss_func,  # the function we want to minimize
        norm_fn=normalize_v,
        loss_constants={"rho": rho},
        loss_kwargs={
            "left_sites": left_sites,
            "proj_sites": proj_sites,
            "right_sites": right_sites,
        },
        autodiff_backend="jax",
        optimizer="L-BFGS-B",
        progbar=False
    )

    v_opt = tnopt.optimize(n=2000)

    v_opt = embed_fp_even_vector(v_opt).contract()

    return v_opt, tnopt.losses

In [39]:
def projector_state_check(state, rho, left_sites):
    left_right_rho = (
        rho
        & state
        & state.conj().reindex({s: f'b{s[1:]}' for s in state.inds})
    )

    left_right_rho = left_right_rho.contract()

    left_inds = [
        f'{s}{i}'
        for i in left_sites
        for s in 'kb'
    ]

    schmidt_decomp = qtn.tensor_core.tensor_split(
        left_right_rho,
        left_inds=left_inds,
        method='svd',
        #cutoff=1e-6,
        cutoff_mode='abs',
        absorb=None,
        renorm=False,
        bond_ind='v'
    )

    schmidt_vals = schmidt_decomp.tensors[1]

    return schmidt_vals

### Embed vector

In [40]:
np_CX = (
    tensor_product_operators(np_00, np_I)
    + tensor_product_operators(np_11, np_X)
)

In [41]:
qu_CX = qtn.Tensor(
    np_CX,
    inds=['k1', 'b1', 'k2', 'b2']
)

In [42]:
np_up_Z_state = np.array([1,0])

In [43]:
qu_up_Z_state = qtn.Tensor(
    data=np_up_Z_state,
    inds=('k',),
    tags='Z0_pad'
)

In [44]:
def embed_fp_even_vector(v):
    # Take a vector of length 2N-1, and return a vector of length 2N
    # which commutes with IZIZ...IZIZ
    # Assuming v site ordering is spin-fermion-spin-...-fermion
    # I think we need at least two fermion sites for this to be reasonable

    sites = sorted(int(s[1:]) for s in v.inds)

    padded_site = sites[-1] + 1

    padded_v = (
        v
        & qu_up_Z_state.reindex({'k': f'k{padded_site}_0'})
    )
    num_cx_gates = len(sites)//2

    cx_gates = [
        qu_CX.reindex({
            'k1': f'k{sites[2*i+1]}',
            'b1': f'b{sites[2*i+1]}',
            'k2': f'k{padded_site}_{i+1}',
            'b2': f'k{padded_site}_{i}'
        })
        for i in range(num_cx_gates-1)
    ]

    i = num_cx_gates-1
    cx_gates.append(
        qu_CX.reindex({
            'k1': f'k{sites[2*i+1]}',
            'b1': f'b{sites[2*i+1]}',
            'k2': f'k{padded_site}',
            'b2': f'k{padded_site}_{i}'
        })
    )

    reindexed_padded_v = (
        padded_v
        .reindex({
            f'k{sites[2*i+1]}': f'b{sites[2*i+1]}'
            for i in range(num_cx_gates)
        })
    )
    sym_v = qtn.TensorNetwork([
        reindexed_padded_v,
        *cx_gates
    ])

    sym_v.mangle_inner_()

    return sym_v

### Loss function

In [45]:
def get_rho_purity(rho, sites):
    # Assuming rho is a Hermitian reduced density matrix
    reindex_map = (
        {f'k{i}': f'b{i}' for i in sites}
        | {f'b{i}': f'k{i}' for i in sites}
    )

    rho_other = rho.reindex(reindex_map)
    #rho_other.mangle_inner_()
    
    out = (rho & rho_other).contract()

    return out

In [134]:
def loss_func(v, rho, left_sites, proj_sites, right_sites):
    embed_v = embed_fp_even_vector(v)
    
    rho_lr = (
        rho
        & embed_v.reindex({f'k{i}': f'b{i}' for i in proj_sites})
        & embed_v.conj()
    )
    rho_lr = rho_lr.contract()
    
    rho_l = rho_lr.reindex(
        {f'k{i}': f'b{i}' for i in right_sites}
    )
    rho_l = rho_l.contract()

    rho_r = rho_lr.reindex(
        {f'k{i}': f'b{i}' for i in left_sites}
    )
    rho_r = rho_r.contract()

    purity_lr = get_rho_purity(rho_lr, left_sites + right_sites)
    purity_l = get_rho_purity(rho_l, left_sites)
    purity_r = get_rho_purity(rho_l, right_sites)

    reindex_map = (
        {f'k{i}': f'b{i}' for i in left_sites+right_sites}
        | {f'b{i}': f'k{i}' for i in left_sites+right_sites}
    )

    cross_term = (
        rho_lr.reindex(reindex_map)
        & rho_l
        & rho_r
    )
    cross_term = cross_term.contract()

    out = jnp.real(
        (purity_lr + purity_l*purity_r - 2*cross_term)/
        purity_lr
    )

    return out

In [47]:
def normalize_v(v):
    norm = (v & v.conj()).contract()
    w = v*jnp.power(norm, -0.5)
    return w

## Extract cut rho and EDM

In [48]:
def generate_edm_from_cut_state(cut_state, sites, num_defect_sites):
    # Lots of duplicate code, probably a better way to do this.
    assert 2*num_defect_sites < len(sites)
    assert (num_defect_sites%2) == 0
    assert (len(sites)%2) == 0
    assert (sites[0]%2) == 0

    left_defect_sites = sites[:num_defect_sites]
    right_defect_sites = sites[-num_defect_sites:]
    internal_sites = sites[num_defect_sites:-num_defect_sites]

    # Being sloppy with the gate indices as the symmetries are invariant
    # under transpose and conjugation.
    left_sym_gates = [
        qu_M.reindex({
            'ks': f'b{i}',
            'kf': f'b{i+1}',
            'bs': f'c{i}',
            'bf': f'c{i+1}'
        })
        for i in left_defect_sites[::2]
    ]

    inner_gates = [
        qu_M.reindex({
            'ks': f'k{i}',
            'kf': f'k{i+1}',
            'bs': f'b{i}',
            'bf': f'b{i+1}'
        })
        for i in internal_sites[::2]
    ]

    right_sym_gates = [
        qu_M.reindex({
            'ks': f'b{i}',
            'kf': f'b{i+1}',
            'bs': f'c{i}',
            'bf': f'c{i+1}'
        })
        for i in right_defect_sites[::2]
    ]

    reindex_map = (
        {
            f'k{i}': f'c{i}'
            for i in (left_defect_sites + right_defect_sites)
        }
        |
        {
            f'k{i}': f'b{i}'
            for i in internal_sites
        }
    )

    # The fact that we don't conjugate the reindexed cut_state means that
    # we are effectively implementing local complex conjugation.
    edm = (
        cut_state
        & cut_state.reindex(reindex_map)
        & left_sym_gates
        & inner_gates
        & right_sym_gates
    )

    edm = edm.contract()

    fuse_maps = [
        ('k_left', (f'k{i}' for i in left_defect_sites)),
        ('b_left', (f'b{i}' for i in left_defect_sites)),
        ('k_right', (f'k{i}' for i in right_defect_sites)),
        ('b_right', (f'b{i}' for i in right_defect_sites))
    ]

    edm.fuse(fuse_maps, inplace=True)

    return edm

## Defect operators

In [49]:
def random_uniform_complex(shape):
    return np.random.uniform(size=shape) + 1j*np.random.uniform(size=shape)

In [50]:
def solve_for_boundary_operators(edm, num_iters=100):
    # Careful, the indices are reversed here for ease.
    # i.e. the k, b indices have been swapped to make tensor contraction easier.
    scores = list()

    u_left = qtn.tensor_builder.rand_tensor(
        (edm.ind_size('b_left'), edm.ind_size('k_left')),
        inds=['k_left', 'b_left'],
        dtype='complex64'
    )

    u_right = qtn.tensor_builder.rand_tensor(
        (edm.ind_size('b_right'), edm.ind_size('k_right')),
        inds=['k_right', 'b_right'],
        dtype='complex64'
    )

    for _ in range(num_iters):
        right_edm = (edm & u_left).contract()
        data = right_edm.data
        U, S, VH = np.linalg.svd(data)
        scores.append(np.sum(S))
    
        sol = (U @ VH).conj().T
        u_right = qtn.Tensor(sol, inds = ['b_right', 'k_right'])

        left_edm = (edm & u_right).contract()
        data = left_edm.data
        U, S, VH = np.linalg.svd(data)
        scores.append(np.sum(S))
    
        sol = (U @ VH).conj().T
        u_left = qtn.Tensor(sol, inds = ['b_left', 'k_left'])

    return (u_left, u_right), scores

## Apply random unitary to groundstate

In [51]:
def generate_random_su2():
    # Randomly sample a unitary, and scale by the determinant.
    u = unitary_group.rvs(2)
    det_u = np.linalg.det(u)
    su = u*np.power(det_u, -0.5)

    return su

In [52]:
X =  generate_random_su2()

In [53]:
np.linalg.det(X)

np.complex128(0.9999999999999999-1.35162183004984e-16j)

In [54]:
np.round(X @ (X.conj().T), 3)

array([[1.+0.j, 0.+0.j],
       [0.+0.j, 1.+0.j]])

In [55]:
def generate_random_symmetry_respecting_unitary_no_offset():
    # Generate a unitary which commutes with MK, where
    # M = X tensor (|0><0|) + Y tensor (|1><1|)

    phi = np.random.uniform(0, 2*np.pi)
    phi_phasor = np.exp(1j*phi)
    u0 = np.diag([phi_phasor, phi_phasor.conj()])

    if random() > 0.5:
        u0 = u0 @ np_X

    u1 = generate_random_su2()

    u = (
        tensor_product_operators(u0, np_00)
        + tensor_product_operators(u1, np_11)
    )

    qu_u = qtn.Tensor(
        u,
        inds=['ks', 'bs', 'kf', 'bf']
    )
    
    return qu_u

In [56]:
def generate_random_symmetry_respecting_unitary_offset():
    # Generate a unitary U such that IUI commutes with (MM)K, where
    # M = X tensor (|0><0|) + Y tensor (|1><1|), and concatenation denotes
    # tensor product.
    phi = np.random.uniform(0, 2*np.pi)
    phi_phasor = np.exp(1j*phi)
    u0 = np.diag([phi_phasor, phi_phasor.conj()])

    phi = np.random.uniform(0, 2*np.pi)
    phi_phasor = np.exp(1j*phi)
    u1 = np.diag([phi_phasor, phi_phasor.conj()])

    u = (
        tensor_product_operators(np_00, u0)
        + tensor_product_operators(np_11, u1)
    )

    qu_u = qtn.Tensor(
        u,
        inds=['kf', 'bf', 'ks', 'bs']
    )
    
    return qu_u

In [57]:
def generate_random_symmetry_respecting_unitary(offset):
    if offset:
        return generate_random_symmetry_respecting_unitary_offset()
    else:
        return generate_random_symmetry_respecting_unitary_no_offset()

In [58]:
# Warning, likely making assupmtions about shape of psi, number of sites being even here etc.
def apply_haar_random_fdlu_to_quimb_state(psi, domains_dict):
    num_sites = domains_dict['num_system_sites']

    depth = domains_dict['fdlu_depth']
    offset = domains_dict['fdlu_offset']
    all_circuit_lists = [
        list() for _ in range(depth)
    ]

    for layer, circuit_list in enumerate(all_circuit_lists):
        delta = layer
        is_offset = ((offset + delta)%2 == 1)

        for i in range(num_sites//2):
            site_1 = ((2*i)+delta+offset)%num_sites
            site_2 = ((2*i)+1+delta+offset)%num_sites

            u = generate_random_symmetry_respecting_unitary(is_offset)

            if is_offset:
                reindex_map = {
                    'kf': f'k_{layer+1}_{site_1}',
                    'ks': f'k_{layer+1}_{site_2}',
                    'bf': f'k_{layer}_{site_1}',
                    'bs': f'k_{layer}_{site_2}'
                }
            else:
                reindex_map = {
                    'ks': f'k_{layer+1}_{site_1}',
                    'kf': f'k_{layer+1}_{site_2}',
                    'bs': f'k_{layer}_{site_1}',
                    'bf': f'k_{layer}_{site_2}'
                }
            
            qu_u = u.reindex(reindex_map)

            circuit_list.append(qu_u)

    all_tensors = (
        [psi.reindex({f'k{i}': f'k_0_{i}' for i in range(num_sites)})]
        + sum(all_circuit_lists, start=[])
    )

    out = (
        qtn
        .TensorNetwork(all_tensors, virtual=False)
        .mangle_inner_()
        .reindex({f'k_{depth}_{i}': f'k{i}' for i in range(num_sites)}) 
    )

    return out

In [59]:
def extract_time_reversal_information_after_random_fdlu(psi, domains_dict,
    num_random_states=20):

    out = list()

    for _ in range(num_random_states):
        rand_psi = apply_haar_random_fdlu_to_quimb_state(psi, domains_dict)
        out.append(extract_time_reversal_information(rand_psi, domains_dict))

    return out

In [60]:
def extract_factorization_time_reversal_information_after_random_fdlu(psi,
    domains_dict, num_random_states=20):
    out = list()

    for _ in range(num_random_states):
        rand_psi = apply_haar_random_fdlu_to_quimb_state(psi, domains_dict)
        data = extract_factorization_time_reversal_information(
            rand_psi,
            domains_dict
        )
        out.append(data)

    return out

In [61]:
def get_quimb_psi_from_quspin_psi(quspin_psi):
    quimb_psi = qtn.Tensor(
        quspin_psi[::-1].reshape((2,)*num_sites),
        inds=[f'k{i}' for i in range(num_sites)]
    )

    return quimb_psi

## Sweep function

In [62]:
def extract_projector(psi, rho_sites, num_pad_sites, jw_even=False):
    proj_sites = rho_sites
    left_sites = list(range(
        min(rho_sites) - num_pad_sites,
        min(rho_sites)
    ))
    right_sites = list(range(
        max(rho_sites)+1,
        max(rho_sites)+num_pad_sites+1
    ))
    all_sites = left_sites + proj_sites + right_sites
    rho = (psi & psi.conj().reindex({f'k{i}': f'b{i}' for i in all_sites}))
    proj_vec, losses = maximize_projector_states(
        rho,
        left_sites,
        proj_sites,
        right_sites
    )

    # This is wrong, assumes that we're working with the left side.
    # So be careful of the right side schmidt_vals, likely wrong.
    schmidt_vals = projector_state_check(
        proj_vec,
        rho,
        left_sites
    )

    return proj_vec, losses, schmidt_vals

In [63]:
def find_invariants_via_projectors_from_random_state(psi, domains_dict, jw_even=False):
    rand_psi = apply_haar_random_fdlu_to_quimb_state(psi, domains_dict)

    left_proj_vec, *left_proj_vec_results = extract_projector(
        rand_psi,
        domains_dict['left_projector_sites'],
        domains_dict['num_projector_pad_sites'],
        jw_even
    )
    
    right_proj_vec, *right_proj_vec_results = extract_projector(
        rand_psi,
        domains_dict['right_projector_sites'],
        domains_dict['num_projector_pad_sites'],
        jw_even
    )
    
    cut_sites = list(range(
        min(domains_dict['left_projector_sites']),
        max(domains_dict['right_projector_sites'])+1
    ))
    
    cut_rho_unprojected = (
        rand_psi
        & rand_psi.conj().reindex({f'k{i}': f'b{i}' for i in cut_sites})
    )
    
    left_proj_sites = domains_dict['left_projector_sites']
    right_proj_sites = domains_dict['right_projector_sites']
    
    cut_rho = (
        cut_rho_unprojected.reindex({
            **{f'k{i}': f'l{i}' for i in left_proj_sites},
            **{f'k{i}': f'l{i}' for i in right_proj_sites},
            **{f'b{i}': f'c{i}' for i in left_proj_sites},
            **{f'b{i}': f'c{i}' for i in right_proj_sites}
        })
        & left_proj_vec.conj().reindex({f'k{i}': f'l{i}' for i in left_proj_sites})
        & left_proj_vec
        & left_proj_vec.reindex({f'k{i}': f'c{i}' for i in left_proj_sites})
        & left_proj_vec.conj().reindex({f'k{i}': f'b{i}' for i in left_proj_sites})
        & right_proj_vec.conj().reindex({f'k{i}': f'l{i}' for i in right_proj_sites})
        & right_proj_vec
        & right_proj_vec.reindex({f'k{i}': f'c{i}' for i in right_proj_sites})
        & right_proj_vec.conj().reindex({f'k{i}': f'b{i}' for i in right_proj_sites})
    )
    
    cut_rho_trace = (
        cut_rho
        .reindex({f'b{i}': f'k{i}' for i in cut_sites})
        .contract()
    )
    
    cut_rho = cut_rho/cut_rho_trace
    
    tranpose_map = (
        {f'k{i}': f'b{i}' for i in cut_sites}
        | {f'b{i}': f'k{i}' for i in cut_sites}
    )
    
    cut_rho_purity = (
        (cut_rho & cut_rho.reindex(tranpose_map))
        .contract()
    )
    
    sub_cut_sites = list(range(
        max(domains_dict['left_projector_sites'])+1,
        min(domains_dict['right_projector_sites'])
    ))
    
    sub_cut_rho = (
        cut_rho
        .reindex({f'k{i}': f'b{i}' for i in left_proj_sites + right_proj_sites})
        .contract()
    )
    
    sub_cut_rho_trace = (
        sub_cut_rho
        .reindex({f'b{i}': f'k{i}' for i in sub_cut_sites})
        .contract()
    )
    
    tranpose_map = (
        {f'k{i}': f'b{i}' for i in sub_cut_sites}
        | {f'b{i}': f'k{i}' for i in sub_cut_sites}
    )
    
    sub_cut_rho_purity = (
        (sub_cut_rho & sub_cut_rho.reindex(tranpose_map))
        .contract()
    )
    
    cut_score, cut_state, cut_overlap = get_dominant_eigenvector(sub_cut_rho, jw_even)

    edm = generate_edm_from_cut_state(
        cut_state,
        sub_cut_sites,
        domains_dict['num_defect_sites']
    )

    defect_ops_results = solve_for_boundary_operators(
        edm,
        num_iters=20
    )

    left_defect_sites = list(range(
        max(domains_dict['left_projector_sites']) + 1,
        max(domains_dict['left_projector_sites']) + 1 + domains_dict['num_defect_sites']
    ))
    right_defect_sites = list(range(
        min(domains_dict['right_projector_sites']) - domains_dict['num_defect_sites'],
        min(domains_dict['right_projector_sites'])
    ))
    
    left_rdm = (
        cut_rho
        .reindex({
            f'b{i}': f'k{i}'
            for i in cut_sites if i not in left_defect_sites
        })
    )
    left_rdm = left_rdm.contract()
    left_fuse_map = [
        ('k_left', [f'k{i}' for i in left_defect_sites]),
        ('b_left', [f'b{i}' for i in left_defect_sites])
    ]
    left_rdm.fuse(left_fuse_map, inplace=True)
    
    right_rdm = (
        cut_rho
        .reindex({
            f'b{i}': f'k{i}'
            for i in cut_sites if i not in right_defect_sites
        })
    )
    right_rdm = right_rdm.contract()
    right_fuse_map = [
        ('k_right', [f'k{i}' for i in right_defect_sites]),
        ('b_right', [f'b{i}' for i in right_defect_sites])
    ]
    right_rdm.fuse(right_fuse_map, inplace=True)

    left_defect_op, right_defect_op = defect_ops_results[0]

    np_left_rdm = (
        left_rdm
        .transpose('k_left', 'b_left')
        .data
    )

    np_left_defect_op = (
        left_defect_op
        .transpose('k_left', 'b_left')
        .data
        .T
    )

    fp_list = [np_I, np_Z]
    np_fp = multikron([
        fp_list[i%2]
        for i in range(domains_dict['num_defect_sites'])
    ])

    left_defect_op_invariant = np.trace(
        np_fp
        @ np_left_defect_op.conj().T
        @ np_fp
        @ np_left_defect_op
        @ np_left_rdm
    )

    np_right_rdm = (
        right_rdm
        .transpose('k_right', 'b_right')
        .data
    )
    
    np_right_defect_op = (
        right_defect_op
        .transpose('k_right', 'b_right')
        .data
        .T
    )

    right_defect_op_invariant = np.trace(
        np_fp
        @ np_right_defect_op.conj().T
        @ np_fp
        @ np_right_defect_op
        @ np_right_rdm
    )

    out = {
        'left_proj_vec': left_proj_vec,
        'left_proj_vec_results': left_proj_vec_results,
        'right_proj_vec': right_proj_vec,
        'right_proj_vec_results': right_proj_vec_results,
        'cut_rho_purity': cut_rho_purity,
        'sub_cut_rho_trace': sub_cut_rho_trace,
        'sub_cut_rho_purity': sub_cut_rho_purity,
        'cut_score': cut_score,
        'cut_state': cut_state,
        'cut_overlap': cut_overlap,
        'defect_ops_scores': defect_ops_results[1],
        'left_defect_op': left_defect_op,
        'right_defect_op': right_defect_op,
        'left_defect_op_invariant': left_defect_op_invariant,
        'right_defect_op_invariant': right_defect_op_invariant
    }

    return out

## Find dominant eigenvector

In [64]:
def random_uniform_complex(shape):
    return np.random.uniform(size=shape) + 1j*np.random.uniform(size=shape)

In [65]:
def lanczos_iteration(rho, v, sites):
    w = (
        rho & v.reindex({f'k{i}': f'b{i}' for i in sites})
    ).contract()

    w_norm = np.sqrt((w & w.conj()).contract())

    out = w/w_norm

    return out

In [66]:
def multiply_state_by_jw_string(state, sites):
    gates = [
        qu_Z.reindex({'k': f'k{i}', 'b': f'b{i}'})
        for i in sites if (i%2 == 1)
    ]

    out = qtn.TensorNetwork(
        [
            *gates,
            state.reindex({f'k{i}': f'b{i}' for i in sites if (i%2 == 1)})
        ]
    )

    return out.contract()

In [67]:
def lanczos_iteration_jw_even(rho, v, sites):
    w = multiply_state_by_jw_string(v, sites)
    w = (v+w)/2

    w = (
        rho & w.reindex({f'k{i}': f'b{i}' for i in sites})
    ).contract()

    w = (multiply_state_by_jw_string(w, sites) + w)/2

    w_norm = np.sqrt((w & w.conj()).contract())

    out = w/w_norm

    return out

In [68]:
def lanczos_algorithm(rho, v, sites, num_iters=20, jw_even=False):
    update_func = lanczos_iteration_jw_even if jw_even else lanczos_iteration
    for _ in range(num_iters):
        v = update_func(rho, v, sites)

    return v

In [69]:
def get_dominant_eigenvector(rho, jw_even=False):
    k_inds = [i for i in rho.inds if i.startswith('k')]
    #b_inds = [i for i in rho.inds if i.startswith('b')]

    sites = [int(s[1:]) for s in k_inds]

    v = qtn.Tensor(
        data=random_uniform_complex((2,)*len(sites)),
        inds=[f'k{i}' for i in sites]
    )

    v = lanczos_algorithm(
        rho,
        v,
        sites,
        num_iters=20,
        jw_even=jw_even
    )

    new_v = lanczos_iteration(rho, v, sites)

    overlap = np.abs((v & new_v.conj()).contract())

    score = (
        v.reindex({f'k{i}': f'b{i}' for i in sites})
        & rho
        & v.conj()
    )
    score = score.contract()


    return score, v, overlap

# Check random unitaries and symmetry

In [70]:
domains_dict = {
    'num_system_sites': 32,
    'left_projector_sites': list(range(6, 10)),
    'right_projector_sites': list(range(22, 26)),
    'num_projector_pad_sites': 2,
    'num_defect_sites': 2,
    'fdlu_depth': 2,
    'fdlu_offset': 1
}

In [71]:
cluster_psi = get_cluster_state_qu_tensor_network(domains_dict['num_system_sites'])

In [72]:
rand_psi = apply_haar_random_fdlu_to_quimb_state(cluster_psi, domains_dict)

In [73]:
(rand_psi & rand_psi.conj()).contract()

np.complex128(0.9999999999999966+2.0816681711721685e-17j)

In [74]:
(rand_psi & cluster_psi.conj()).contract()

np.complex128(-4.994603773567405e-06+1.0323214044661785e-21j)

In [75]:
symmetry_gates = [
    qu_M.reindex({
        'ks': f'k{i}', 'bs':f'b{i}',
        'kf': f'k{i+1}', 'bf':f'b{i+1}',
    })
    for i in range(0, domains_dict['num_system_sites'], 2)
]

In [76]:
sym_cluster_psi = qtn.TensorNetwork(
    [
        rand_psi.reindex({f'k{i}': f'b{i}' for i in range(domains_dict['num_system_sites'])}),
        *symmetry_gates
    ]
)

In [77]:
(
    sym_cluster_psi & sym_cluster_psi.conj()
).contract()

np.complex128(0.9999999999999966+2.0816681711721685e-17j)

In [78]:
(
    sym_cluster_psi & rand_psi.conj()
).contract()

np.complex128(-0.0002986315857543319-9.486769009248164e-20j)

In [79]:
(
    sym_cluster_psi & rand_psi
).contract()

np.complex128(0.9999999999999963+6.938893903907228e-17j)

In [80]:
product_psi = get_product_qu_tensor_network(domains_dict['num_system_sites'])

In [81]:
rand_psi = apply_haar_random_fdlu_to_quimb_state(product_psi, domains_dict)

In [82]:
(rand_psi & rand_psi.conj()).contract()

np.complex128(0.9999999999999959+6.938893903907228e-17j)

In [83]:
(rand_psi & product_psi.conj()).contract()

np.complex128(-1.2644594622403106e-05+1.449956669375141e-21j)

In [84]:
symmetry_gates = [
    qu_M.reindex({
        'ks': f'k{i}', 'bs':f'b{i}',
        'kf': f'k{i+1}', 'bf':f'b{i+1}',
    })
    for i in range(0, domains_dict['num_system_sites'], 2)
]

In [85]:
sym_product_psi = qtn.TensorNetwork(
    [
        rand_psi.reindex({f'k{i}': f'b{i}' for i in range(domains_dict['num_system_sites'])}),
        *symmetry_gates
    ]
)

In [86]:
(
    sym_product_psi & sym_product_psi.conj()
).contract()

np.complex128(0.9999999999999959+6.938893903907228e-17j)

In [87]:
(
    sym_product_psi & rand_psi.conj()
).contract()

np.complex128(4.955450197358733e-05-1.3806637040245096e-19j)

In [88]:
(
    sym_product_psi & rand_psi
).contract()

np.complex128(0.999999999999996+6.938893903907228e-17j)

So random unitaries are symmetric, nice.

# Test - Cluster state

In [226]:
domains_dict = {
    'num_system_sites': 32,
    'left_projector_sites': list(range(6, 10)),
    'right_projector_sites': list(range(22, 26)),
    'num_projector_pad_sites': 2,
    'num_defect_sites': 2,
    'fdlu_depth': 0,
    'fdlu_offset': 0
}

In [227]:
cluster_psi = get_cluster_state_qu_tensor_network(domains_dict['num_system_sites'])

In [248]:
results = list()

for _ in tqdm(range(50)):
    current = find_invariants_via_projectors_from_random_state(
        cluster_psi,
        domains_dict,
        jw_even=True
    )
    results.append(current)

100%|███████████████████████████████████████████████████████████████████████| 50/50 [02:26<00:00,  2.94s/it]


### Analyze results

#### Projector scores

In [249]:
np.array([
    d['left_proj_vec_results'][0][-1]
    for d in results
])

array([0.56250113, 0.56250125, 0.56250113, 0.56250113, 0.56250107,
       0.56250113, 0.56250113, 0.56250113, 0.56250113, 0.56250107,
       0.56250107, 0.56250113, 0.56250113, 0.56250107, 0.56250125,
       0.56250113, 0.56250113, 0.56250113, 0.56250119, 0.56250113,
       0.56250113, 0.56250107, 0.56250125, 0.56250125, 0.56250107,
       0.56250113, 0.56250113, 0.56250107, 0.56250113, 0.56250113,
       0.56250119, 0.56250107, 0.56250113, 0.56250125, 0.56250107,
       0.56250125, 0.56250101, 0.56250107, 0.56250113, 0.56250113,
       0.56250113, 0.56250101, 0.56250113, 0.56250113, 0.56250107,
       0.56250101, 0.56250119, 0.56250113, 0.56250107, 0.56250113])

I thought this should be zero...

In [250]:
np.array([
    d['right_proj_vec_results'][0][-1]
    for d in results
])

array([0.56250125, 0.56250107, 0.56250113, 0.56250107, 0.56250107,
       0.56250113, 0.56250107, 0.56250107, 0.56250113, 0.56250113,
       0.56250113, 0.56250113, 0.56250113, 0.56250107, 0.56250107,
       0.56250113, 0.56250107, 0.56250107, 0.56250107, 0.56250101,
       0.56250113, 0.56250113, 0.56250113, 0.56250113, 0.56250125,
       0.56250113, 0.56250125, 0.56250113, 0.56250107, 0.56250113,
       0.56250113, 0.56250113, 0.56250113, 0.56250113, 0.56250113,
       0.56250113, 0.56250113, 0.56250101, 0.56250107, 0.56250113,
       0.56250107, 0.56250113, 0.56250113, 0.56250101, 0.56250119,
       0.56250113, 0.56250113, 0.56250107, 0.56250113, 0.56250113])

In [251]:
def get_schmidt_vals_ratio(schmidt_vals):
    if len(schmidt_vals.data) > 1:
        return schmidt_vals.data[1]/schmidt_vals.data[0]
    else:
        return 0

In [252]:
np.round(np.array([
    get_schmidt_vals_ratio(d['left_proj_vec_results'][1])
    for d in results
]), 3)

array([0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
       0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
       0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.])

In [253]:
np.round(np.array([
    get_schmidt_vals_ratio(d['right_proj_vec_results'][1])
    for d in results
]), 3)

array([0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
       0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
       0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.])

#### Purities

In [254]:
np.round(np.array(
    [d['cut_rho_purity'] for d in results]
), 3)

array([1.+0.j, 1.-0.j, 1.+0.j, 1.-0.j, 1.+0.j, 1.-0.j, 1.+0.j, 1.-0.j,
       1.+0.j, 1.+0.j, 1.-0.j, 1.+0.j, 1.+0.j, 1.-0.j, 1.-0.j, 1.+0.j,
       1.+0.j, 1.-0.j, 1.-0.j, 1.+0.j, 1.+0.j, 1.-0.j, 1.+0.j, 1.+0.j,
       1.-0.j, 1.-0.j, 1.+0.j, 1.-0.j, 1.+0.j, 1.-0.j, 1.-0.j, 1.+0.j,
       1.+0.j, 1.-0.j, 1.-0.j, 1.-0.j, 1.+0.j, 1.+0.j, 1.+0.j, 1.+0.j,
       1.-0.j, 1.+0.j, 1.-0.j, 1.-0.j, 1.+0.j, 1.-0.j, 1.-0.j, 1.+0.j,
       1.+0.j, 1.+0.j])

In [255]:
np.round(np.array(
    [d['sub_cut_rho_trace'] for d in results]
), 3)

array([1.+0.j, 1.-0.j, 1.-0.j, 1.+0.j, 1.+0.j, 1.-0.j, 1.+0.j, 1.-0.j,
       1.+0.j, 1.+0.j, 1.+0.j, 1.+0.j, 1.+0.j, 1.-0.j, 1.-0.j, 1.+0.j,
       1.+0.j, 1.-0.j, 1.-0.j, 1.+0.j, 1.+0.j, 1.-0.j, 1.+0.j, 1.-0.j,
       1.-0.j, 1.-0.j, 1.-0.j, 1.-0.j, 1.+0.j, 1.-0.j, 1.-0.j, 1.+0.j,
       1.+0.j, 1.-0.j, 1.+0.j, 1.-0.j, 1.+0.j, 1.-0.j, 1.+0.j, 1.+0.j,
       1.-0.j, 1.-0.j, 1.-0.j, 1.-0.j, 1.+0.j, 1.-0.j, 1.-0.j, 1.+0.j,
       1.+0.j, 1.-0.j])

In [256]:
np.round(np.array(
    [d['sub_cut_rho_purity'] for d in results]
), 3)

array([1.+0.j, 1.-0.j, 1.-0.j, 1.+0.j, 1.+0.j, 1.-0.j, 1.+0.j, 1.-0.j,
       1.+0.j, 1.+0.j, 1.+0.j, 1.+0.j, 1.+0.j, 1.-0.j, 1.-0.j, 1.+0.j,
       1.+0.j, 1.-0.j, 1.-0.j, 1.+0.j, 1.+0.j, 1.-0.j, 1.+0.j, 1.-0.j,
       1.-0.j, 1.-0.j, 1.-0.j, 1.-0.j, 1.+0.j, 1.-0.j, 1.-0.j, 1.+0.j,
       1.+0.j, 1.-0.j, 1.+0.j, 1.-0.j, 1.+0.j, 1.-0.j, 1.+0.j, 1.+0.j,
       1.-0.j, 1.-0.j, 1.-0.j, 1.-0.j, 1.+0.j, 1.-0.j, 1.-0.j, 1.+0.j,
       1.+0.j, 1.-0.j])

Purities are the same, which one could likely prove.

#### Cut state results

In [257]:
np.round(np.array(
    [d['cut_score'] for d in results]
), 3)

array([0.-0.j, 1.+0.j, 1.-0.j, 0.+0.j, 1.+0.j, 0.-0.j, 1.+0.j, 0.+0.j,
       1.-0.j, 1.+0.j, 0.-0.j, 1.-0.j, 1.-0.j, 0.+0.j, 0.+0.j, 1.-0.j,
       1.-0.j, 0.-0.j, 0.-0.j, 1.+0.j, 0.+0.j, 0.+0.j, 1.-0.j, 0.-0.j,
       0.+0.j, 1.+0.j, 0.-0.j, 1.-0.j, 1.-0.j, 1.-0.j, 0.-0.j, 0.+0.j,
       1.+0.j, 1.+0.j, 1.+0.j, 0.-0.j, 0.+0.j, 0.-0.j, 0.+0.j, 0.+0.j,
       0.+0.j, 1.-0.j, 0.+0.j, 0.-0.j, 1.-0.j, 0.-0.j, 1.-0.j, 0.+0.j,
       0.+0.j, 1.+0.j])

In [258]:
np.round(np.array(
    [d['cut_overlap'] for d in results]
), 3)

array([1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1.,
       1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1.,
       1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1.])

#### Defect op scores overlaps

In [259]:
np.round(np.array([d['defect_ops_scores'][-1] for d in results]), 5)

array([1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1.,
       1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1.,
       1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1.])

In [260]:
np.array([d['defect_ops_scores'][-1] for d in results])

array([1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1.,
       1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1.,
       1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1.])

In [261]:
[d['defect_ops_scores'][-1] for d in results]

[np.float64(1.0000000000000004),
 np.float64(1.0000000000000009),
 np.float64(1.0),
 np.float64(1.0),
 np.float64(1.0),
 np.float64(1.0000000000000002),
 np.float64(1.0000000000000004),
 np.float64(0.9999999999999999),
 np.float64(0.9999999999999998),
 np.float64(1.0),
 np.float64(1.0),
 np.float64(1.0000000000000002),
 np.float64(1.0),
 np.float64(0.9999999999999998),
 np.float64(0.9999999999999997),
 np.float64(1.0000000000000002),
 np.float64(1.0),
 np.float64(1.0),
 np.float64(1.0000000000000002),
 np.float64(1.0000000000000002),
 np.float64(1.0000000000000004),
 np.float64(1.0),
 np.float64(1.0000000000000004),
 np.float64(1.0),
 np.float64(0.9999999999999998),
 np.float64(1.0),
 np.float64(0.9999999999999999),
 np.float64(1.0),
 np.float64(1.0),
 np.float64(1.0),
 np.float64(1.0),
 np.float64(1.0),
 np.float64(1.0000000000000004),
 np.float64(1.0000000000000002),
 np.float64(1.0),
 np.float64(1.0),
 np.float64(1.0000000000000004),
 np.float64(0.9999999999999998),
 np.float64(1.00

#### Phases

In [262]:
left_phases = np.array([
    d['left_defect_op_invariant'] for d in results
])

In [263]:
left_phases.shape

(50,)

In [264]:
np.round(left_phases, 3)

array([ 1.-0.j, -1.+0.j, -1.-0.j, -1.-0.j, -1.-0.j, -1.+0.j, -1.-0.j,
        1.-0.j, -1.-0.j, -1.-0.j,  1.-0.j, -1.-0.j, -1.+0.j,  1.-0.j,
       -1.-0.j, -1.-0.j, -1.-0.j, -1.+0.j, -1.+0.j, -1.-0.j, -1.-0.j,
        1.-0.j, -1.-0.j,  1.+0.j, -1.-0.j, -1.+0.j,  1.+0.j, -1.+0.j,
       -1.-0.j, -1.-0.j, -1.+0.j,  1.-0.j, -1.-0.j, -1.+0.j, -1.+0.j,
       -1.+0.j, -1.-0.j,  1.+0.j, -1.-0.j, -1.-0.j, -1.+0.j, -1.+0.j,
       -1.+0.j, -1.-0.j, -1.+0.j, -1.+0.j, -1.+0.j,  1.-0.j, -1.-0.j,
       -1.-0.j])

Weird things happening when cut_score = 0. Why is this happening?

In [265]:
right_phases = np.array([
    d['right_defect_op_invariant'] for d in results
])

In [266]:
right_phases.shape

(50,)

In [267]:
np.round(right_phases, 3)

array([-1.-0.j, -1.+0.j, -1.+0.j, -1.+0.j, -1.-0.j, -1.+0.j, -1.-0.j,
       -1.+0.j, -1.-0.j, -1.-0.j, -1.+0.j, -1.-0.j, -1.-0.j, -1.-0.j,
       -1.-0.j, -1.-0.j, -1.-0.j, -1.+0.j, -1.+0.j, -1.-0.j, -1.+0.j,
       -1.+0.j, -1.-0.j, -1.-0.j, -1.+0.j, -1.+0.j, -1.-0.j, -1.+0.j,
       -1.-0.j, -1.+0.j, -1.+0.j, -1.-0.j, -1.-0.j, -1.+0.j, -1.+0.j,
       -1.+0.j, -1.-0.j, -1.+0.j, -1.-0.j, -1.-0.j, -1.+0.j, -1.+0.j,
       -1.+0.j, -1.+0.j, -1.-0.j, -1.+0.j, -1.+0.j, -1.+0.j, -1.+0.j,
       -1.+0.j])

## Debug
What's happening in the first case? Unrwap.

In [280]:
left_proj_vec = results[0]['left_proj_vec']

In [273]:
rho_sites = domains_dict['left_projector_sites']
num_pad_sites = domains_dict['num_projector_pad_sites']

In [274]:
proj_sites = rho_sites
left_sites = list(range(
    min(rho_sites) - num_pad_sites,
    min(rho_sites)
))
right_sites = list(range(
    max(rho_sites)+1,
    max(rho_sites)+num_pad_sites+1
))
all_sites = left_sites + proj_sites + right_sites

In [276]:
rho = (cluster_psi & cluster_psi.conj().reindex({f'k{i}': f'b{i}' for i in all_sites}))

In [281]:
left_proj_vec

Tensor(shape=(2, 2, 2, 2), inds=('k6', 'k8', 'k7', 'k9'), tags=oset(['Z0_pad']))

In [282]:
embed_v = left_proj_vec

In [283]:
rho_lr = (
    rho
    & embed_v.reindex({f'k{i}': f'b{i}' for i in proj_sites})
    & embed_v.conj()
)
rho_lr = rho_lr.contract()

rho_l = rho_lr.reindex(
    {f'k{i}': f'b{i}' for i in right_sites}
)
rho_l = rho_l.contract()

rho_r = rho_lr.reindex(
    {f'k{i}': f'b{i}' for i in left_sites}
)
rho_r = rho_r.contract()

purity_lr = get_rho_purity(rho_lr, left_sites + right_sites)
purity_l = get_rho_purity(rho_l, left_sites)
purity_r = get_rho_purity(rho_l, right_sites)

reindex_map = (
    {f'k{i}': f'b{i}' for i in left_sites+right_sites}
    | {f'b{i}': f'k{i}' for i in left_sites+right_sites}
)

cross_term = (
    rho_lr.reindex(reindex_map)
    & rho_l
    & rho_r
)
cross_term = cross_term.contract()

In [284]:
purity_lr

np.complex128(0.015624999663347607-1.7347234576893284e-18j)

In [285]:
purity_l

np.complex128(0.03124999927941945-3.469446911953358e-18j)

In [286]:
purity_r

np.complex128(0.03124999927941945-3.469446911953358e-18j)

In [287]:
cross_term

np.complex128(0.0039062498648911475-6.505212884912067e-19j)

In [288]:
numerator = (
    purity_lr
    + purity_l*purity_r
    - 2*cross_term
)

In [289]:
numerator

np.complex128(0.008789062388529029-6.505213077039678e-19j)

In [290]:
numerator/purity_lr

np.complex128(0.5625000049853441+2.0816681785757344e-17j)

# Test - Random FDLU cluster state - depth 1

In [158]:
domains_dict = {
    'num_system_sites': 32,
    'left_projector_sites': list(range(4, 10)),
    'right_projector_sites': list(range(22, 28)),
    'num_projector_pad_sites': 2,
    'num_defect_sites': 2,
    'fdlu_depth': 1,
    'fdlu_offset': 0
}

In [159]:
cluster_psi = get_cluster_state_qu_tensor_network(domains_dict['num_system_sites'])

In [160]:
results = list()

for _ in tqdm(range(20)):
    current = find_invariants_via_projectors_from_random_state(
        cluster_psi,
        domains_dict,
        jw_even=True
    )
    results.append(current)

100%|███████████████████████████████████████████████████████████████████████| 20/20 [00:58<00:00,  2.91s/it]


### Analyze results

#### Projector scores

In [181]:
np.array([
    d['left_proj_vec_results'][0][-1]
    for d in results
])

array([0.53755063, 0.55933493, 0.53137225, 0.55972344, 0.56039017,
       0.55980146, 0.55617809, 0.54184991, 0.56249988, 0.55851686,
       0.55655843, 0.5367955 , 0.55197108, 0.54195583, 0.55787665,
       0.55545938, 0.54956234, 0.53941292, 0.56193161, 0.56231827])

In [182]:
np.array([
    d['right_proj_vec_results'][0][-1]
    for d in results
])

array([0.56213474, 0.55879676, 0.56167954, 0.53156757, 0.55967063,
       0.55625445, 0.56025529, 0.55403215, 0.54764199, 0.55978864,
       0.55567038, 0.56249923, 0.56230825, 0.5384655 , 0.53387153,
       0.5553714 , 0.55456108, 0.53622872, 0.55588847, 0.56187254])

In [183]:
def get_schmidt_vals_ratio(schmidt_vals):
    if len(schmidt_vals.data) > 1:
        return schmidt_vals.data[1]/schmidt_vals.data[0]
    else:
        return 0

In [184]:
np.round(np.array([
    get_schmidt_vals_ratio(d['left_proj_vec_results'][1])
    for d in results
]), 3)

array([0.   , 0.   , 0.   , 0.   , 0.001, 0.   , 0.   , 0.   , 0.   ,
       0.   , 0.   , 0.001, 0.   , 0.   , 0.   , 0.   , 0.   , 0.   ,
       0.   , 0.   ])

In [185]:
np.round(np.array([
    get_schmidt_vals_ratio(d['right_proj_vec_results'][1])
    for d in results
]), 3)

array([0.   , 0.   , 0.   , 0.   , 0.   , 0.001, 0.   , 0.   , 0.   ,
       0.   , 0.   , 0.   , 0.   , 0.   , 0.   , 0.   , 0.   , 0.   ,
       0.   , 0.   ])

#### Purities

In [186]:
np.round(np.array(
    [d['cut_rho_purity'] for d in results]
), 3)

array([1.+0.j, 1.+0.j, 1.-0.j, 1.+0.j, 1.+0.j, 1.-0.j, 1.-0.j, 1.-0.j,
       1.-0.j, 1.+0.j, 1.-0.j, 1.-0.j, 1.+0.j, 1.-0.j, 1.-0.j, 1.+0.j,
       1.+0.j, 1.+0.j, 1.+0.j, 1.+0.j])

In [187]:
np.round(np.array(
    [d['sub_cut_rho_trace'] for d in results]
), 3)

array([1.-0.j, 1.+0.j, 1.-0.j, 1.+0.j, 1.+0.j, 1.-0.j, 1.-0.j, 1.-0.j,
       1.-0.j, 1.+0.j, 1.+0.j, 1.+0.j, 1.+0.j, 1.+0.j, 1.-0.j, 1.+0.j,
       1.+0.j, 1.+0.j, 1.+0.j, 1.+0.j])

In [188]:
np.round(np.array(
    [d['sub_cut_rho_purity'] for d in results]
), 3)

array([1.-0.j, 1.+0.j, 1.-0.j, 1.+0.j, 1.+0.j, 1.-0.j, 1.-0.j, 1.-0.j,
       1.-0.j, 1.+0.j, 1.+0.j, 1.+0.j, 1.+0.j, 1.+0.j, 1.-0.j, 1.-0.j,
       1.+0.j, 1.+0.j, 1.+0.j, 1.+0.j])

Purities are the same, which one could likely prove.

#### Cut state results

In [189]:
np.round(np.array(
    [d['cut_score'] for d in results]
), 3)

array([1.-0.j, 1.+0.j, 1.-0.j, 0.+0.j, 0.-0.j, 1.-0.j, 1.-0.j, 1.-0.j,
       0.-0.j, 1.+0.j, 1.+0.j, 0.+0.j, 1.+0.j, 1.+0.j, 0.-0.j, 0.+0.j,
       1.+0.j, 0.+0.j, 0.+0.j, 1.+0.j])

In [190]:
np.round(np.array(
    [d['cut_overlap'] for d in results]
), 3)

array([1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1.,
       1., 1., 1.])

#### Defect op scores overlaps

In [191]:
np.round(np.array([d['defect_ops_scores'][-1] for d in results]), 5)

array([1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1.,
       1., 1., 1.])

In [192]:
np.array([d['defect_ops_scores'][-1] for d in results])

array([1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1.,
       1., 1., 1.])

In [193]:
[d['defect_ops_scores'][-1] for d in results]

[np.float64(1.0000000000000004),
 np.float64(0.9999999999999998),
 np.float64(0.9999999999999998),
 np.float64(1.0000000000000004),
 np.float64(0.9999999999999996),
 np.float64(1.0000000000000004),
 np.float64(0.9999999999999996),
 np.float64(1.0),
 np.float64(1.0000000000000002),
 np.float64(1.0000000000000004),
 np.float64(0.9999999999999999),
 np.float64(1.0000000000000002),
 np.float64(0.9999999999999993),
 np.float64(1.0000000000000002),
 np.float64(1.0),
 np.float64(1.0),
 np.float64(1.0),
 np.float64(0.9999999999999999),
 np.float64(0.9999999999999999),
 np.float64(1.0)]

#### Phases

In [194]:
left_phases = np.array([
    d['left_defect_op_invariant'] for d in results
])

In [195]:
left_phases.shape

(20,)

In [196]:
left_phases

array([-1.        +6.59194921e-17j, -0.99999998-1.38777875e-17j,
       -1.        +5.46437895e-17j, -1.        -4.16333637e-17j,
        0.96586043+2.68080123e-17j, -1.        +5.55111512e-17j,
       -1.        +1.28369537e-16j, -1.        +5.11743425e-17j,
       -1.        -4.16333634e-17j, -1.        -5.55111512e-17j,
       -1.        +8.65870958e-17j, -0.91623704-7.47618854e-17j,
       -1.        -3.46944695e-17j, -0.99999995+1.38777870e-17j,
       -1.        -6.93889390e-17j, -1.        +2.08166819e-17j,
       -1.        +1.00613962e-16j, -1.        -1.83880688e-16j,
       -1.        -3.12250225e-17j, -0.99999997-3.03793448e-16j])

All good, apart from the last one. What's happening there?

In [197]:
right_phases = np.array([
    d['right_defect_op_invariant'] for d in results
])

In [198]:
right_phases.shape

(20,)

In [199]:
right_phases

array([-1.        +2.42861287e-17j, -1.        -7.63278329e-17j,
       -0.99999999+1.92554306e-16j, -0.99773489-5.53854157e-17j,
       -1.        +5.20417043e-17j, -1.        +7.48357728e-26j,
       -0.99999999-1.38777879e-17j, -0.99999989-3.70346117e-24j,
       -1.        +7.63278331e-17j, -0.99999996-8.32667249e-17j,
       -1.        +9.02056208e-17j, -1.        -2.21189477e-26j,
       -0.99999996+5.46437895e-17j, -0.99999998-5.19266335e-26j,
        0.73635605+7.36463700e-17j, -1.        +6.76542155e-17j,
       -1.        -2.77555756e-17j, -1.        -9.02056208e-17j,
        0.39673197-2.48893122e-17j, -0.99999999-4.16333632e-17j])

# Test - Random FDLU cluster state - depth 2

In [200]:
domains_dict = {
    'num_system_sites': 32,
    'left_projector_sites': list(range(4, 10)),
    'right_projector_sites': list(range(22, 28)),
    'num_projector_pad_sites': 2,
    'num_defect_sites': 2,
    'fdlu_depth': 2,
    'fdlu_offset': 0
}

In [201]:
cluster_psi = get_cluster_state_qu_tensor_network(domains_dict['num_system_sites'])

In [205]:
results = list()

for _ in tqdm(range(20)):
    current = find_invariants_via_projectors_from_random_state(
        cluster_psi,
        domains_dict,
        jw_even=True
    )
    results.append(current)

100%|███████████████████████████████████████████████████████████████████████| 20/20 [01:53<00:00,  5.65s/it]


### Analyze results

#### Projector scores

In [206]:
np.array([
    d['left_proj_vec_results'][0][-1]
    for d in results
])

array([0.68236852, 0.58925068, 0.60869545, 0.57036179, 0.60711831,
       0.68176544, 0.65892404, 0.64333618, 0.59904891, 0.60452545,
       0.67499596, 0.69248563, 0.67805773, 0.58875418, 0.59709799,
       0.72305566, 0.59786904, 0.75941843, 0.5656895 , 0.56476706])

In [207]:
np.array([
    d['right_proj_vec_results'][0][-1]
    for d in results
])

array([0.66742063, 0.61542869, 0.56220526, 0.73915339, 0.73465258,
       0.76355714, 0.55749285, 0.67697942, 0.67283821, 0.60340595,
       0.66980284, 0.55928111, 0.5976615 , 0.69921535, 0.63401645,
       0.68801254, 0.58633649, 0.70679063, 0.61278093, 0.71012062])

In [208]:
def get_schmidt_vals_ratio(schmidt_vals):
    if len(schmidt_vals.data) > 1:
        return schmidt_vals.data[1]/schmidt_vals.data[0]
    else:
        return 0

In [209]:
np.round(np.array([
    get_schmidt_vals_ratio(d['left_proj_vec_results'][1])
    for d in results
]), 3)

array([0.122, 0.199, 0.003, 0.011, 0.411, 0.089, 0.1  , 0.091, 0.006,
       0.04 , 0.071, 0.14 , 0.078, 0.144, 0.247, 0.   , 0.139, 0.029,
       0.016, 0.035])

In [210]:
np.round(np.array([
    get_schmidt_vals_ratio(d['right_proj_vec_results'][1])
    for d in results
]), 3)

array([0.063, 0.006, 0.002, 0.002, 0.16 , 0.142, 0.   , 0.213, 0.038,
       0.053, 0.171, 0.002, 0.066, 0.019, 0.426, 0.11 , 0.032, 0.102,
       0.405, 0.092])

#### Purities

In [211]:
np.round(np.array(
    [d['cut_rho_purity'] for d in results]
), 3)

array([1.+0.j, 1.-0.j, 1.-0.j, 1.+0.j, 1.+0.j, 1.-0.j, 1.-0.j, 1.+0.j,
       1.+0.j, 1.-0.j, 1.-0.j, 1.+0.j, 1.-0.j, 1.-0.j, 1.+0.j, 1.+0.j,
       1.-0.j, 1.-0.j, 1.-0.j, 1.-0.j])

In [212]:
np.round(np.array(
    [d['sub_cut_rho_trace'] for d in results]
), 3)

array([1.+0.j, 1.-0.j, 1.-0.j, 1.-0.j, 1.+0.j, 1.-0.j, 1.-0.j, 1.+0.j,
       1.+0.j, 1.-0.j, 1.-0.j, 1.+0.j, 1.+0.j, 1.-0.j, 1.+0.j, 1.+0.j,
       1.-0.j, 1.-0.j, 1.-0.j, 1.-0.j])

In [213]:
np.round(np.array(
    [d['sub_cut_rho_purity'] for d in results]
), 3)

array([1.+0.j, 1.-0.j, 1.-0.j, 1.-0.j, 1.+0.j, 1.-0.j, 1.-0.j, 1.+0.j,
       1.+0.j, 1.-0.j, 1.-0.j, 1.+0.j, 1.+0.j, 1.-0.j, 1.+0.j, 1.+0.j,
       1.-0.j, 1.-0.j, 1.-0.j, 1.-0.j])

Purities are the same, which one could likely prove.

#### Cut state results

In [214]:
np.round(np.array(
    [d['cut_score'] for d in results]
), 3)

array([0.-0.j, 1.-0.j, 1.-0.j, 0.-0.j, 1.+0.j, 0.+0.j, 0.-0.j, 0.+0.j,
       1.+0.j, 1.+0.j, 0.-0.j, 1.+0.j, 0.+0.j, 0.+0.j, 0.-0.j, 1.+0.j,
       1.-0.j, 0.-0.j, 1.-0.j, 1.-0.j])

In [215]:
np.round(np.array(
    [d['cut_overlap'] for d in results]
), 3)

array([1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1.,
       1., 1., 1.])

#### Defect op scores overlaps

In [216]:
np.round(np.array([d['defect_ops_scores'][-1] for d in results]), 5)

array([0.93342, 0.74943, 0.5062 , 0.24992, 0.30058, 0.13159, 0.93716,
       0.63467, 0.57302, 0.48022, 0.62382, 0.47423, 0.32011, 0.76799,
       0.93453, 0.64406, 0.84695, 0.5739 , 0.93303, 0.72133])

In [217]:
np.array([d['defect_ops_scores'][-1] for d in results])

array([0.93342327, 0.74943039, 0.50620452, 0.24992041, 0.30057663,
       0.13159415, 0.93716093, 0.63467201, 0.5730183 , 0.48022023,
       0.62381716, 0.47423216, 0.32010919, 0.76798881, 0.93453029,
       0.64405911, 0.84694541, 0.57389668, 0.93302622, 0.72133137])

In [218]:
[d['defect_ops_scores'][-1] for d in results]

[np.float64(0.9334232723725817),
 np.float64(0.7494303932052457),
 np.float64(0.50620451550843),
 np.float64(0.2499204109588752),
 np.float64(0.3005766311795034),
 np.float64(0.1315941529412597),
 np.float64(0.9371609301361857),
 np.float64(0.6346720087562288),
 np.float64(0.5730183028748715),
 np.float64(0.4802202310679649),
 np.float64(0.6238171645909917),
 np.float64(0.4742321620836377),
 np.float64(0.3201091868913539),
 np.float64(0.7679888124984211),
 np.float64(0.9345302884448745),
 np.float64(0.6440591064337392),
 np.float64(0.8469454097946529),
 np.float64(0.5738966760644304),
 np.float64(0.9330262205549446),
 np.float64(0.7213313686147862)]

#### Phases

In [219]:
left_phases = np.array([
    d['left_defect_op_invariant'] for d in results
])

In [220]:
left_phases.shape

(20,)

In [221]:
left_phases

array([-1.        +3.22953053e-17j, -0.99999999+8.18074196e-17j,
       -1.        -1.73811239e-17j, -1.        +6.79724467e-17j,
       -1.        -1.29479101e-17j, -1.        -9.11066464e-17j,
       -1.        +6.57462251e-17j, -0.99999999-6.29192458e-17j,
       -0.99999999-4.80270501e-17j, -0.99999999-1.82015520e-16j,
        0.25336243-1.20888542e-17j, -0.99999971-4.69744844e-17j,
       -1.        +3.67636740e-17j, -0.99999995-3.54606549e-17j,
       -1.        -1.41315548e-16j, -0.99999998-1.50365610e-16j,
       -1.        +8.08568797e-17j, -1.        -3.36655372e-18j,
       -1.        +5.95471760e-17j, -1.        -1.44937166e-16j])

In [223]:
right_phases = np.array([
    d['right_defect_op_invariant'] for d in results
])

In [224]:
right_phases.shape

(20,)

In [225]:
right_phases

array([-1.        -4.98571377e-17j, -0.86158945+9.52400829e-17j,
       -0.95291764+1.20270905e-16j, -0.96840839+7.23819855e-17j,
       -0.99254055-1.65530647e-17j, -0.99999985+1.18737722e-16j,
       -1.        +3.23593739e-16j,  0.98642823+7.23802055e-17j,
       -0.98308667+4.22036306e-19j, -0.71648146+2.69352105e-17j,
       -0.99832953+6.16339735e-17j, -0.96388525+2.71599239e-17j,
       -1.        -5.55092389e-17j,  0.80744756-8.27219720e-17j,
       -1.        -1.36838425e-16j, -0.73291707+2.25853846e-17j,
       -0.94795447+1.40694015e-16j, -0.99999978+5.45898706e-17j,
       -0.92766638+1.17779066e-16j, -0.88009607+3.20193134e-17j])

# Test - product state

In [294]:
domains_dict = {
    'num_system_sites': 32,
    'left_projector_sites': list(range(6, 10)),
    'right_projector_sites': list(range(22, 26)),
    'num_projector_pad_sites': 2,
    'num_defect_sites': 2,
    'fdlu_depth': 0,
    'fdlu_offset': 0
}

In [295]:
product_psi = get_product_qu_tensor_network(domains_dict['num_system_sites'])

In [293]:
results = list()

for _ in tqdm(range(50)):
    current = find_invariants_via_projectors_from_random_state(
        product_psi,
        domains_dict,
        jw_even=True
    )
    results.append(current)

100%|███████████████████████████████████████████████████████████████████████| 50/50 [02:59<00:00,  3.60s/it]


### Analyze results

#### Projector scores

In [296]:
np.array([
    d['left_proj_vec_results'][0][-1]
    for d in results
])

array([ 3.57958783e-07,  2.38674744e-07, -1.57743847e-19,  1.19282291e-07,
       -1.78330140e-20,  1.19287790e-07,  1.19283257e-07, -1.77003610e-19,
       -9.66681004e-20,  2.38656469e-07,  2.38674431e-07,  1.19315075e-07,
        2.38611676e-07,  3.57961852e-07,  2.38571545e-07,  1.19332782e-07,
       -5.82124543e-20,  2.38665308e-07,  3.58014006e-07,  1.19315288e-07,
        1.19283314e-07,  1.19282973e-07,  2.38658544e-07,  2.38638478e-07,
       -1.57206534e-21,  2.38637227e-07,  1.19317406e-07, -1.70763101e-19,
        1.19281523e-07,  2.38650387e-07, -1.53188770e-19,  3.57963927e-07,
        2.38706633e-07,  2.38686397e-07,  1.19316979e-07,  2.38626171e-07,
        1.19300736e-07,  2.38611506e-07,  2.38625319e-07,  1.19281950e-07,
       -2.79492955e-19,  1.19331276e-07,  3.58016820e-07,  1.19330480e-07,
        1.19330679e-07,  2.38672925e-07,  1.19309647e-07,  1.19309419e-07,
        2.38610795e-07,  1.19306534e-07])

In [297]:
np.array([
    d['right_proj_vec_results'][0][-1]
    for d in results
])

array([ 3.58004144e-07,  1.19280671e-07,  1.19330622e-07,  3.58001643e-07,
        3.58016649e-07,  2.38656071e-07,  1.19283513e-07,  2.38609715e-07,
        1.19283314e-07,  3.58034129e-07,  1.19280983e-07,  1.19300566e-07,
        1.19283484e-07, -2.59224911e-19,  1.19272727e-07,  3.58002779e-07,
        1.19270538e-07,  3.58016990e-07,  2.38638307e-07, -1.19209588e-07,
       -1.44447888e-19,  3.58005110e-07,  3.58007213e-07,  3.57964012e-07,
        1.19330736e-07,  2.38655815e-07,  3.57959806e-07,  2.38618355e-07,
        1.19281125e-07,  2.38667866e-07,  3.58006446e-07,  1.19281893e-07,
        3.58015967e-07,  1.19310016e-07,  2.38681110e-07,  3.58014944e-07,
        2.38608976e-07,  2.38626939e-07,  3.58001728e-07,  2.38674346e-07,
        1.19330764e-07, -8.76826959e-20,  3.58008180e-07,  1.19294242e-07,
        1.19308595e-07,  2.38625802e-07,  1.19281410e-07,  2.38648454e-07,
        1.19326941e-07,  1.19274148e-07])

In [298]:
def get_schmidt_vals_ratio(schmidt_vals):
    if len(schmidt_vals.data) > 1:
        return schmidt_vals.data[1]/schmidt_vals.data[0]
    else:
        return 0

In [299]:
np.round(np.array([
    get_schmidt_vals_ratio(d['left_proj_vec_results'][1])
    for d in results
]), 3)

array([0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
       0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
       0, 0, 0, 0, 0, 0])

In [300]:
np.round(np.array([
    get_schmidt_vals_ratio(d['right_proj_vec_results'][1])
    for d in results
]), 3)

array([0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
       0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
       0, 0, 0, 0, 0, 0])

#### Purities

In [301]:
np.round(np.array(
    [d['cut_rho_purity'] for d in results]
), 3)

array([1.-0.j, 1.+0.j, 1.+0.j, 1.+0.j, 1.-0.j, 1.+0.j, 1.+0.j, 1.-0.j,
       1.+0.j, 1.+0.j, 1.+0.j, 1.+0.j, 1.-0.j, 1.+0.j, 1.-0.j, 1.+0.j,
       1.-0.j, 1.-0.j, 1.-0.j, 1.-0.j, 1.+0.j, 1.+0.j, 1.+0.j, 1.-0.j,
       1.-0.j, 1.-0.j, 1.+0.j, 1.+0.j, 1.-0.j, 1.-0.j, 1.+0.j, 1.+0.j,
       1.+0.j, 1.+0.j, 1.+0.j, 1.+0.j, 1.-0.j, 1.-0.j, 1.+0.j, 1.-0.j,
       1.-0.j, 1.-0.j, 1.-0.j, 1.-0.j, 1.+0.j, 1.+0.j, 1.+0.j, 1.+0.j,
       1.-0.j, 1.-0.j])

In [302]:
np.round(np.array(
    [d['sub_cut_rho_trace'] for d in results]
), 3)

array([1.-0.j, 1.-0.j, 1.+0.j, 1.+0.j, 1.-0.j, 1.+0.j, 1.+0.j, 1.-0.j,
       1.+0.j, 1.+0.j, 1.-0.j, 1.+0.j, 1.-0.j, 1.+0.j, 1.+0.j, 1.-0.j,
       1.-0.j, 1.-0.j, 1.-0.j, 1.+0.j, 1.+0.j, 1.+0.j, 1.-0.j, 1.-0.j,
       1.-0.j, 1.+0.j, 1.-0.j, 1.+0.j, 1.-0.j, 1.-0.j, 1.+0.j, 1.+0.j,
       1.-0.j, 1.+0.j, 1.+0.j, 1.+0.j, 1.-0.j, 1.-0.j, 1.+0.j, 1.+0.j,
       1.+0.j, 1.-0.j, 1.-0.j, 1.-0.j, 1.+0.j, 1.+0.j, 1.-0.j, 1.-0.j,
       1.+0.j, 1.-0.j])

In [303]:
np.round(np.array(
    [d['sub_cut_rho_purity'] for d in results]
), 3)

array([1.-0.j, 1.-0.j, 1.+0.j, 1.+0.j, 1.-0.j, 1.+0.j, 1.+0.j, 1.-0.j,
       1.+0.j, 1.+0.j, 1.-0.j, 1.+0.j, 1.-0.j, 1.+0.j, 1.+0.j, 1.-0.j,
       1.-0.j, 1.-0.j, 1.-0.j, 1.+0.j, 1.+0.j, 1.+0.j, 1.-0.j, 1.-0.j,
       1.-0.j, 1.+0.j, 1.-0.j, 1.+0.j, 1.-0.j, 1.-0.j, 1.+0.j, 1.+0.j,
       1.-0.j, 1.+0.j, 1.+0.j, 1.+0.j, 1.-0.j, 1.-0.j, 1.+0.j, 1.+0.j,
       1.+0.j, 1.-0.j, 1.-0.j, 1.-0.j, 1.+0.j, 1.+0.j, 1.-0.j, 1.-0.j,
       1.+0.j, 1.-0.j])

Purities are the same, which one could likely prove.

#### Cut state results

In [304]:
np.round(np.array(
    [d['cut_score'] for d in results]
), 3)

array([1.-0.j, 1.-0.j, 1.+0.j, 1.-0.j, 1.-0.j, 1.-0.j, 1.-0.j, 1.+0.j,
       1.-0.j, 1.+0.j, 1.-0.j, 1.+0.j, 1.-0.j, 1.+0.j, 1.-0.j, 1.+0.j,
       1.+0.j, 1.+0.j, 1.+0.j, 1.+0.j, 1.+0.j, 1.+0.j, 1.+0.j, 1.-0.j,
       1.+0.j, 1.-0.j, 1.+0.j, 1.-0.j, 1.+0.j, 1.-0.j, 1.+0.j, 1.+0.j,
       1.+0.j, 1.+0.j, 1.-0.j, 1.-0.j, 1.+0.j, 1.-0.j, 1.-0.j, 1.+0.j,
       1.-0.j, 1.-0.j, 1.-0.j, 1.-0.j, 1.+0.j, 1.-0.j, 1.-0.j, 1.-0.j,
       1.+0.j, 1.+0.j])

In [305]:
np.round(np.array(
    [d['cut_overlap'] for d in results]
), 3)

array([1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1.,
       1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1.,
       1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1.])

#### Defect op scores overlaps

In [306]:
np.round(np.array([d['defect_ops_scores'][-1] for d in results]), 5)

array([1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1.,
       1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1.,
       1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1.])

In [307]:
np.array([d['defect_ops_scores'][-1] for d in results])

array([1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1.,
       1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1.,
       1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1.])

In [308]:
[d['defect_ops_scores'][-1] for d in results]

[np.float64(1.0000000000000007),
 np.float64(1.0),
 np.float64(1.0000000000000007),
 np.float64(1.0000000000000002),
 np.float64(0.9999999999999998),
 np.float64(1.0000000000000002),
 np.float64(1.0),
 np.float64(1.0),
 np.float64(0.9999999999999997),
 np.float64(1.0000000000000002),
 np.float64(1.0),
 np.float64(1.0000000000000004),
 np.float64(0.9999999999999997),
 np.float64(0.9999999999999996),
 np.float64(0.9999999999999996),
 np.float64(0.9999999999999999),
 np.float64(1.0000000000000007),
 np.float64(1.0),
 np.float64(0.9999999999999997),
 np.float64(0.9999999999999997),
 np.float64(1.0000000000000004),
 np.float64(0.9999999999999994),
 np.float64(1.0000000000000002),
 np.float64(1.0000000000000004),
 np.float64(0.9999999999999994),
 np.float64(1.0),
 np.float64(0.9999999999999997),
 np.float64(1.0000000000000002),
 np.float64(1.0000000000000002),
 np.float64(0.9999999999999998),
 np.float64(1.0000000000000002),
 np.float64(1.0000000000000002),
 np.float64(1.0000000000000002),
 

#### Phases

In [309]:
left_phases = np.array([
    d['left_defect_op_invariant'] for d in results
])

In [310]:
left_phases.shape

(50,)

In [311]:
np.round(left_phases, 3)

array([1.-0.j, 1.-0.j, 1.-0.j, 1.-0.j, 1.-0.j, 1.+0.j, 1.+0.j, 1.-0.j,
       1.+0.j, 1.+0.j, 1.-0.j, 1.+0.j, 1.-0.j, 1.+0.j, 1.+0.j, 1.-0.j,
       1.-0.j, 1.-0.j, 1.-0.j, 1.+0.j, 1.+0.j, 1.+0.j, 1.+0.j, 1.+0.j,
       1.-0.j, 1.-0.j, 1.-0.j, 1.+0.j, 1.-0.j, 1.+0.j, 1.-0.j, 1.+0.j,
       1.-0.j, 1.+0.j, 1.-0.j, 1.+0.j, 1.-0.j, 1.-0.j, 1.-0.j, 1.+0.j,
       1.-0.j, 1.-0.j, 1.+0.j, 1.+0.j, 1.-0.j, 1.+0.j, 1.-0.j, 1.-0.j,
       1.+0.j, 1.+0.j])

In [314]:
right_phases = np.array([
    d['right_defect_op_invariant'] for d in results
])

In [315]:
right_phases.shape

(50,)

In [316]:
np.round(right_phases, 3)

array([1.-0.j, 1.-0.j, 1.-0.j, 1.-0.j, 1.-0.j, 1.+0.j, 1.+0.j, 1.-0.j,
       1.+0.j, 1.+0.j, 1.-0.j, 1.+0.j, 1.-0.j, 1.+0.j, 1.+0.j, 1.-0.j,
       1.-0.j, 1.-0.j, 1.-0.j, 1.+0.j, 1.+0.j, 1.+0.j, 1.+0.j, 1.+0.j,
       1.-0.j, 1.-0.j, 1.-0.j, 1.+0.j, 1.-0.j, 1.+0.j, 1.-0.j, 1.+0.j,
       1.-0.j, 1.+0.j, 1.-0.j, 1.+0.j, 1.-0.j, 1.-0.j, 1.-0.j, 1.+0.j,
       1.-0.j, 1.-0.j, 1.+0.j, 1.+0.j, 1.-0.j, 1.+0.j, 1.-0.j, 1.-0.j,
       1.+0.j, 1.+0.j])

# Sweep
## Product state

In [ ]:
domains_dict = {
    'num_system_sites': 32,
    'left_projector_sites': list(range(6, 10)),
    'right_projector_sites': list(range(22, 26)),
    'num_projector_pad_sites': 2,
    'num_defect_sites': 2,
    'fdlu_depth': 2,
    'fdlu_offset': 0
}

In [ ]:
product_psi = get_product_qu_tensor_network(domains_dict['num_system_sites'])

In [ ]:
results = list()

for _ in tqdm(range(50)):
    current = find_invariants_via_projectors_from_random_state(product_psi, domains_dict)
    results.append(current)

### Analyze results

#### Projector scores

In [ ]:
np.round(np.array([
    d['left_proj_vec_results'][0]
    for d in results
]), 3)

In [ ]:
np.round(np.array([
    d['right_proj_vec_results'][0]
    for d in results
]), 3)

In [ ]:
np.round(np.array([
    d['left_proj_vec_results'][1]
    for d in results
]), 3)

In [ ]:
np.round(np.array([
    d['right_proj_vec_results'][1]
    for d in results
]), 3)

In [ ]:
def get_schmidt_vals_ratio(schmidt_vals):
    if len(schmidt_vals.data) > 1:
        return schmidt_vals.data[1]/schmidt_vals.data[0]
    else:
        return 0

In [ ]:
np.round(np.array([
    get_schmidt_vals_ratio(d['left_proj_vec_results'][2])
    for d in results
]), 3)

In [ ]:
np.round(np.array([
    get_schmidt_vals_ratio(d['right_proj_vec_results'][2])
    for d in results
]), 3)

#### Purities

In [ ]:
np.round(np.array(
    [d['cut_rho_purity'] for d in results]
), 3)

In [ ]:
np.round(np.array(
    [d['sub_cut_rho_trace'] for d in results]
), 3)

In [ ]:
np.round(np.array(
    [d['sub_cut_rho_purity'] for d in results]
), 3)

Purities are the same, which one could likely prove.

#### Cut state results

In [ ]:
np.round(np.array(
    [d['cut_score'] for d in results]
), 3)

In [ ]:
np.round(np.array(
    [d['cut_overlap'] for d in results]
), 3)

#### Defect op scores overlaps

In [ ]:
np.round(np.array([d['defect_ops_scores'][-1] for d in results]), 5)

In [ ]:
np.array([d['defect_ops_scores'][-1] for d in results])

In [ ]:
[d['defect_ops_scores'][-1] for d in results]

#### Phases

In [ ]:
left_phases = np.array([
    d['left_defect_op_invariant'] for d in results
])

In [ ]:
left_phases.shape

In [ ]:
left_phases

In [ ]:
right_phases = np.array([
    d['right_defect_op_invariant'] for d in results
])

In [ ]:
right_phases.shape

In [ ]:
right_phases

## Cluster state

In [ ]:
domains_dict = {
    'num_system_sites': 32,
    'left_projector_sites': list(range(4, 10)),
    'right_projector_sites': list(range(22, 28)),
    'num_projector_pad_sites': 2,
    'num_defect_sites': 4,
    'fdlu_depth': 2,
    'fdlu_offset': 0
}

In [ ]:
cluster_psi = get_cluster_state_qu_tensor_network(domains_dict['num_system_sites'])

In [ ]:
results = list()

for _ in tqdm(range(20)):
    current = find_invariants_via_projectors_from_random_state(cluster_psi, domains_dict)
    results.append(current)

### Analyze results

#### Projector scores

In [ ]:
np.round(np.array([
    d['left_proj_vec_results'][0]
    for d in results
]), 3)

In [ ]:
np.round(np.array([
    d['right_proj_vec_results'][0]
    for d in results
]), 3)

In [ ]:
np.round(np.array([
    d['left_proj_vec_results'][1]
    for d in results
]), 3)

In [ ]:
np.round(np.array([
    d['right_proj_vec_results'][1]
    for d in results
]), 3)

In [ ]:
def get_schmidt_vals_ratio(schmidt_vals):
    if len(schmidt_vals.data) > 1:
        return schmidt_vals.data[1]/schmidt_vals.data[0]
    else:
        return 0

In [ ]:
np.round(np.array([
    get_schmidt_vals_ratio(d['left_proj_vec_results'][2])
    for d in results
]), 3)

In [ ]:
np.round(np.array([
    get_schmidt_vals_ratio(d['right_proj_vec_results'][2])
    for d in results
]), 3)

#### Purities

In [ ]:
np.round(np.array(
    [d['cut_rho_purity'] for d in results]
), 3)

In [ ]:
np.round(np.array(
    [d['sub_cut_rho_trace'] for d in results]
), 3)

In [ ]:
np.round(np.array(
    [d['sub_cut_rho_purity'] for d in results]
), 3)

Purities are the same, which one could likely prove.

#### Cut state results

In [ ]:
np.round(np.array(
    [d['cut_score'] for d in results]
), 3)

In [ ]:
np.round(np.array(
    [d['cut_overlap'] for d in results]
), 3)

#### Defect op scores overlaps

In [ ]:
np.round(np.array([d['defect_ops_scores'][-1] for d in results]), 5)

In [ ]:
np.array([d['defect_ops_scores'][-1] for d in results])

In [ ]:
[d['defect_ops_scores'][-1] for d in results]

#### Phases

In [ ]:
left_phases = np.array([
    d['left_defect_op_invariant'] for d in results
])

In [ ]:
left_phases.shape

In [ ]:
left_phases

In [ ]:
right_phases = np.array([
    d['right_defect_op_invariant'] for d in results
])

In [ ]:
right_phases.shape

In [ ]:
right_phases

# Old code
## Check projector is FP even.

In [ ]:
results['left_proj_vec']

In [ ]:
v = results['left_proj_vec']

In [ ]:
v

In [ ]:
gates = [
    qu_Z.reindex({'k': f'k{i}', 'b': f'b{i}'})
    for i in [7,9]
]

In [ ]:
v_sym = qtn.TensorNetwork(
    [*gates, v.reindex({f'k{i}': f'b{i}' for i in [7,9]})]
)

In [ ]:
(v & v.conj()).contract()

In [ ]:
(v_sym & v_sym.conj()).contract()

In [ ]:
(v & v_sym.conj()).contract()

No, not working.

In [ ]:
results['right_proj_vec']

In [ ]:
v = results['right_proj_vec']

In [ ]:
v

In [ ]:
gates = [
    qu_Z.reindex({'k': f'k{i}', 'b': f'b{i}'})
    for i in [23,25]
]

In [ ]:
v_sym = qtn.TensorNetwork(
    [*gates, v.reindex({f'k{i}': f'b{i}' for i in [23,25]})]
)

In [ ]:
(v & v.conj()).contract()

In [ ]:
(v_sym & v_sym.conj()).contract()

In [ ]:
(v & v_sym.conj()).contract()

# Test optimization
## Embed vector

In [ ]:
np_CX = (
    tensor_product_operators(np_00, np_I)
    + tensor_product_operators(np_11, np_X)
)

In [ ]:
qu_CX = qtn.Tensor(
    np_CX,
    inds=['k1', 'b1', 'k2', 'b2']
)

In [ ]:
np_up_Z_state = np.array([1,0])

In [ ]:
qu_up_Z_state = qtn.Tensor(
    data=np_up_Z_state,
    inds=('k',),
    tags='Z0_pad'
)

In [ ]:
def embed_fp_even_vector(v):
    # Take a vector of length 2N-1, and return a vector of length 2N
    # which commutes with IZIZ...IZIZ
    # Assuming v site ordering is spin-fermion-spin-...-fermion
    # I think we need at least two fermion sites for this to be reasonable

    sites = sorted(int(s[1:]) for s in v.inds)

    padded_site = sites[-1] + 1

    padded_v = (
        v
        & qu_up_Z_state.reindex({'k': f'k{padded_site}_0'})
    )
    num_cx_gates = len(sites)//2

    cx_gates = [
        qu_CX.reindex({
            'k1': f'k{sites[2*i+1]}',
            'b1': f'b{sites[2*i+1]}',
            'k2': f'k{padded_site}_{i+1}',
            'b2': f'k{padded_site}_{i}'
        })
        for i in range(num_cx_gates-1)
    ]

    i = num_cx_gates-1
    cx_gates.append(
        qu_CX.reindex({
            'k1': f'k{sites[2*i+1]}',
            'b1': f'b{sites[2*i+1]}',
            'k2': f'k{padded_site}',
            'b2': f'k{padded_site}_{i}'
        })
    )

    reindexed_padded_v = (
        padded_v
        .reindex({
            f'k{sites[2*i+1]}': f'b{sites[2*i+1]}'
            for i in range(num_cx_gates)
        })
    )
    sym_v = qtn.TensorNetwork([
        reindexed_padded_v,
        *cx_gates
    ])

    sym_v.mangle_inner_()

    return sym_v

In [ ]:
test_v = qtn.Tensor(
    data=random_uniform_complex((2,)*9),
    inds=[f'k{i}' for i in range(9)]
)

In [ ]:
norm = (test_v & test_v.conj()).contract()

In [ ]:
test_v = test_v*np.power(norm, -0.5)

In [ ]:
(test_v & test_v.conj()).contract()

In [ ]:
sym_v = embed_fp_even_vector(test_v)

In [ ]:
(sym_v & sym_v.conj()).contract()

In [ ]:
sym_v

In [ ]:
sym_v.draw()

In [ ]:
sym_v.outer_inds()

In [ ]:
sym_v = sym_v.contract()

In [ ]:
gates = [
    qu_Z.reindex({'k': f'k{i}', 'b': f'b{i}'})
    for i in range(1, 10, 2)
]

In [ ]:
sym_sym_v = qtn.TensorNetwork(
    [*gates, sym_v.reindex({f'k{i}': f'b{i}' for i in range(1, 10, 2)})]
)

In [ ]:
sym_sym_v

In [ ]:
(sym_sym_v & sym_sym_v.conj()).contract()

In [ ]:
(sym_v & sym_sym_v.conj()).contract()

So the output is symmetric, great.

## Loss function

In [ ]:
def get_rho_purity(rho, sites):
    # Assuming rho is a Hermitian reduced density matrix
    reindex_map = (
        {f'k{i}': f'b{i}' for i in sites}
        | {f'b{i}': f'k{i}' for i in sites}
    )

    rho_other = rho.reindex(reindex_map)
    #rho_other.mangle_inner_()
    
    out = (rho & rho_other).contract()

    return out

In [ ]:
def loss_func(v, rho, left_sites, proj_sites, right_sites):
    embed_v = embed_fp_even_vector(v)
    
    rho_lr = (
        rho
        & embed_v.reindex({f'k{i}': f'b{i}' for i in proj_sites})
        & embed_v.conj()
    )
    rho_lr = rho_lr.contract()
    
    rho_l = rho_lr.reindex(
        {f'k{i}': f'b{i}' for i in right_sites}
    )
    rho_l = rho_l.contract()

    rho_r = rho_lr.reindex(
        {f'k{i}': f'b{i}' for i in left_sites}
    )
    rho_r = rho_r.contract()

    purity_lr = get_rho_purity(rho_lr, left_sites + right_sites)
    purity_l = get_rho_purity(rho_l, left_sites)
    purity_r = get_rho_purity(rho_l, right_sites)

    reindex_map = (
        {f'k{i}': f'b{i}' for i in left_sites+right_sites}
        | {f'b{i}': f'k{i}' for i in left_sites+right_sites}
    )

    cross_term = (
        rho_lr.reindex(reindex_map)
        & rho_l
        & rho_r
    )
    cross_term = cross_term.contract()

    out = jnp.real(purity_lr + purity_l*purity_r - 2*cross_term)

    return out

## Test optimization

In [ ]:
domains_dict = {
    'num_system_sites': 32,
    'left_projector_sites': list(range(6, 10)),
    'right_projector_sites': list(range(22, 26)),
    'num_projector_pad_sites': 2,
    'num_defect_sites': 2,
    'fdlu_depth': 2,
    'fdlu_offset': 1
}

In [ ]:
cluster_psi = get_cluster_state_qu_tensor_network(domains_dict['num_system_sites'])

In [ ]:
left_sites = [14, 15]
proj_sites = [16, 17, 18, 19]
right_sites = [20, 21]

all_sites = (
    left_sites
    + proj_sites
    + right_sites
)

In [ ]:
rho = (
    cluster_psi
    & cluster_psi.conj().reindex({
        f'k{i}': f'b{i}' for i in all_sites
    })
)

In [ ]:
rho.contract()

In [ ]:
v = qtn.Tensor(
    data=random_uniform_complex((2,)*3),
    inds=[f'k{i}' for i in proj_sites[:-1]]
)

In [ ]:
norm = (v & v.conj()).contract()

In [ ]:
v = v*np.power(norm, -0.5)

In [ ]:
(v & v.conj()).contract()

In [ ]:
def normalize_v(v):
    norm = (v & v.conj()).contract()
    w = v*jnp.power(norm, -0.5)
    return w

In [ ]:
loss_constants={"rho": rho}
loss_kwargs={
    "left_sites": left_sites,
    "proj_sites": proj_sites,
    "right_sites": right_sites,
}

In [ ]:
rho.outer_inds()

In [ ]:
all_sites

In [ ]:
get_rho_purity(rho, all_sites)

In [ ]:
v

In [ ]:
embed_v = embed_fp_even_vector(v)

In [ ]:
embed_v.outer_inds()

In [ ]:
embed_v.outer_inds()

In [ ]:
rho_lr = rho_lr = (
    rho
    & embed_v.reindex({f'k{i}': f'b{i}' for i in proj_sites})
    & embed_v.conj()
)

In [ ]:
rho.ind_map['k16']

In [ ]:
embed_v.reindex({f'k{i}': f'b{i}' for i in proj_sites})

In [ ]:
embed_v

In [ ]:
rho_lr.contract()

In [ ]:
rho_lr.ind_map['k16']

In [ ]:
rho_lr.tensors[55]

In [ ]:
rho_lr.tensors[163]

In [ ]:
rho_lr = (
    rho
    & embed_v.reindex({f'k{i}': f'b{i}' for i in proj_sites})
    & embed_v.conj()
)
rho_lr = rho_lr.contract()

rho_l = rho_lr.reindex(
    {f'k{i}': f'b{i}' for i in right_sites}
)
rho_l = rho_l.contract()

rho_r = rho_lr.reindex(
    {f'k{i}': f'b{i}' for i in left_sites}
)
rho_r = rho_r.contract()

In [ ]:
embed_v.outer_inds()

In [ ]:
cross_term = (
    rho_lr.reindex(reindex_map)
    & rho_l
    & rho_r
)
cross_term = cross_term.contract()

In [ ]:
cross_term

In [ ]:
sites = left_sites

In [ ]:
reindex_map = (
    {f'k{i}': f'b{i}' for i in sites}
    | {f'b{i}': f'k{i}' for i in sites}
)

In [ ]:
reindex_map

In [ ]:
rho.reindex(reindex_map).outer_inds()

In [ ]:
rho.outer_inds()

In [ ]:
out = (rho & rho.reindex(reindex_map)).contract()

In [ ]:
get_rho_purity(rho_lr, left_sites+right_sites)

In [ ]:
loss_func(
    v,
    **loss_constants,
    **loss_kwargs
)

In [ ]:
rho_lr.reindex(reindex_map).outer_inds()

In [ ]:
rho_l.outer_inds()

In [ ]:
rho_r.outer_inds()

In [ ]:
tnopt = qtn.TNOptimizer(
    v,  # the tensor network we want to optimize
    loss_func,  # the function we want to minimize
    norm_fn=normalize_v,
    loss_constants={"rho": rho},
    loss_kwargs={
        "left_sites": left_sites,
        "proj_sites": proj_sites,
        "right_sites": right_sites,
    },
    autodiff_backend="jax",
    optimizer="L-BFGS-B",
    progbar=False
)

In [ ]:
v_opt = tnopt.optimize(n=2000)

In [ ]:
v_opt

In [ ]:
plt.plot(tnopt.losses)

In [ ]:
tnopt.losses

Rapid convergence...